# Myllia: Bilinear v2 (Weighted Cosine Loss + Baseline Blending + Optional GenePT)

This notebook extends the bilinear conditional factorization model with three medium-large upgrades:

1) Metric-aware loss: weighted L1-like + weighted cosine loss  
2) Learned amplitude control: blend predictions with delta_baseline using a learned scalar s(g)  
3) Optional GenePT embeddings: if enabled, the notebook tries to download and load GenePT embeddings

Defaults:
- Uses SVD embeddings from training_cells.h5ad for both pert genes and output genes
- GenePT is off by default

If GenePT fails to download or load, it falls back to the SVD embeddings.


In [19]:
# -----------------------------
# Imports and settings
# -----------------------------
import os
import zipfile
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import anndata as ad
import scanpy as sc
from scipy import sparse

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

from myllia_metric import myllia_score

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

ROOT = Path(".")

# Big knobs
RANK_R       = 32
EMB_DIM_PERT = 128
EMB_DIM_OUT  = 128

# Loss mix
LAMBDA_COS   = 0.35
GATE_A       = 0.0
GATE_B       = 0.2
EPS          = 1e-12

# Training
DROPOUT      = 0.10
LR           = 2e-3
WD           = 1e-4
EPOCHS       = 1200
BATCH_GENES  = 16
EVAL_EVERY   = 25
PATIENCE     = 14

# Embeddings
USE_GENEPT = True

# GenePT download (optional)
GENEPT_URL = "https://zenodo.org/records/10833191/files/GenePT_emebdding_v2.zip?download=1"
GENEPT_DIR = ROOT / "external" / "genept"
GENEPT_DIR.mkdir(parents=True, exist_ok=True)

print("device:", device)


device: cuda


In [20]:
# -----------------------------
# Metric scoring helper
# -----------------------------
def score_delta(dt: np.ndarray, dp: np.ndarray) -> dict:
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)
    return {
        "score": float(r.score),
        "wcos": float(r.wcos),
        "mean_term": float(r.mean_term),
        "pred_wmae": float(r.pred_wmae),
    }


In [21]:
# -----------------------------
# Load core competition data
# -----------------------------
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = (X_train_means - x_base[None, :]).astype(np.float32)  # (80, 5127)

delta_baseline = D_train.mean(axis=0).astype(np.float32)

val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))


Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


In [22]:
# -----------------------------
# Embeddings: SVD from training_cells.h5ad
# -----------------------------
def find_h5ad():
    candidates = [
        ROOT / "data" / "training_cells.h5ad",
        ROOT / "Data" / "training_cells.h5ad",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("training_cells.h5ad not found in data/ or Data/")

def build_svd_embeddings_from_h5ad(h5ad_path: Path, union_genes: list[str], k: int, seed: int):
    adata = ad.read_h5ad(str(h5ad_path))
    adata = adata.copy()

    # normalize on ALL genes first
    sc.pp.normalize_total(adata, target_sum=1e4, inplace=True)

    varU = pd.Index([str(v).upper() for v in adata.var_names])
    genesU = [str(g).upper() for g in union_genes]
    pos = varU.get_indexer(genesU)
    ok = pos >= 0

    genes_ok = [genesU[i] for i in range(len(genesU)) if ok[i]]
    pos_ok = pos[ok]

    missing = [genesU[i] for i in range(len(genesU)) if not ok[i]]
    if missing:
        print(f"[warn] {len(missing)} / {len(genesU)} union genes missing from h5ad. Example: {missing[:12]}")

    adata = adata[:, pos_ok].copy()

    X = adata.X
    if not sparse.issparse(X):
        X = sparse.csr_matrix(X)
    else:
        X = X.tocsr(copy=True)

    X.data = np.log2(X.data + 1.0).astype(np.float32)

    svd = TruncatedSVD(n_components=k, random_state=seed)
    svd.fit(X)
    gene_emb = svd.components_.T.astype(np.float32)

    gene2emb = {genes_ok[i]: gene_emb[i] for i in range(len(genes_ok))}
    fallback = gene_emb.mean(axis=0).astype(np.float32)
    return gene2emb, fallback

h5ad_path = find_h5ad()

val_targets = df_valmap["pert"].astype(str).tolist()
union_genes = sorted(set([g.upper() for g in gene_columns] +
                         [g.upper() for g in train_genes.tolist()] +
                         [g.upper() for g in val_targets]))

print("union_genes:", len(union_genes))

gene2emb_svd, emb_fallback_svd = build_svd_embeddings_from_h5ad(
    h5ad_path=h5ad_path,
    union_genes=union_genes,
    k=max(EMB_DIM_PERT, EMB_DIM_OUT),
    seed=SEED
)

def emb_svd(g: str, d: int) -> np.ndarray:
    v = gene2emb_svd.get(str(g).upper(), emb_fallback_svd)
    return v[:d].copy()

print("[ok] built SVD embeddings from h5ad")


union_genes: 5143
[ok] built SVD embeddings from h5ad


In [23]:
# -----------------------------
# Optional: GenePT embeddings (best-effort)
# -----------------------------
def _download(url: str, dst: Path, force: bool = False):
    import urllib.request
    if dst.exists() and not force:
        return
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept": "*/*",
        "Connection": "keep-alive",
    }
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as r, open(dst, "wb") as f:
        f.write(r.read())

def try_load_genept(gene_list: list[str], out_dim: int = 128):
    zip_path = GENEPT_DIR / "GenePT_embedding_v2.zip"
    try:
        print("[genept] downloading...")
        _download(GENEPT_URL, zip_path, force=False)
        print("[genept] downloaded:", zip_path)
    except Exception as e:
        print("[genept] download failed:", repr(e))
        return None, None

    extract_dir = GENEPT_DIR / "extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)
    except Exception as e:
        print("[genept] unzip failed:", repr(e))
        return None, None

    pkl_files = []
    for root, _, files in os.walk(extract_dir):
        for fn in files:
            lo = fn.lower()
            if lo.endswith(".pickle") or lo.endswith(".pkl"):
                pkl_files.append(Path(root) / fn)

    if not pkl_files:
        print("[genept] no pickle found")
        return None, None

    chosen = None
    for p in pkl_files:
        name = p.name.lower()
        if "model_3" in name or "model-3" in name or "m3" in name:
            chosen = p
            break
    if chosen is None:
        chosen = pkl_files[0]

    print("[genept] using pickle:", chosen)

    try:
        with open(chosen, "rb") as f:
            obj = pickle.load(f)
    except Exception as e:
        print("[genept] pickle load failed:", repr(e))
        return None, None

    if not isinstance(obj, dict):
        print("[genept] expected dict gene->vec, got:", type(obj))
        return None, None

    genesU = [str(g).upper() for g in gene_list]
    vecs = []
    ok_genes = []
    for g in genesU:
        v = obj.get(g, None)
        if v is None:
            continue
        v = np.asarray(v, dtype=np.float32).ravel()
        if v.size < out_dim:
            continue
        vecs.append(v[:out_dim])
        ok_genes.append(g)

    if not vecs:
        print("[genept] no overlap between gene list and genept dict")
        return None, None

    M = np.vstack(vecs).astype(np.float32)
    fallback = M.mean(axis=0).astype(np.float32)
    gene2emb = {ok_genes[i]: M[i] for i in range(len(ok_genes))}
    print(f"[genept] coverage: {len(ok_genes)}/{len(genesU)}")
    return gene2emb, fallback

gene2emb_genept, emb_fallback_genept = (None, None)
if USE_GENEPT:
    gene2emb_genept, emb_fallback_genept = try_load_genept(union_genes, out_dim=max(EMB_DIM_PERT, EMB_DIM_OUT))
    if gene2emb_genept is None:
        print("[genept] falling back to SVD embeddings")
        USE_GENEPT = False


[genept] downloading...
[genept] downloaded: external\genept\GenePT_embedding_v2.zip
[genept] using pickle: external\genept\extracted\GenePT_emebdding_v2\GenePT_gene_protein_embedding_model_3_text.pickle
[genept] coverage: 4978/5143


In [24]:
# -----------------------------
# Final embedding accessor
# -----------------------------
def emb(g: str, d: int) -> np.ndarray:
    gu = str(g).upper()
    if USE_GENEPT and gene2emb_genept is not None:
        v = gene2emb_genept.get(gu, emb_fallback_genept)
        return np.asarray(v, np.float32)[:d].copy()
    return emb_svd(gu, d)

U_out = np.vstack([emb(g, EMB_DIM_OUT) for g in gene_columns]).astype(np.float32)
Z_train = np.vstack([emb(g, EMB_DIM_PERT) for g in train_genes]).astype(np.float32)

print("U_out:", U_out.shape, "Z_train:", Z_train.shape, "USE_GENEPT:", USE_GENEPT)


U_out: (5127, 128) Z_train: (80, 128) USE_GENEPT: True


In [25]:
# -----------------------------
# Loss: weighted L1-like + weighted cosine
# -----------------------------
def gate_smoothstep(x: torch.Tensor, a: float = GATE_A, b: float = GATE_B) -> torch.Tensor:
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def weighted_l1_like(dt: torch.Tensor, dp: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    w = gate_smoothstep(torch.abs(dt), a=GATE_A, b=GATE_B)
    err = torch.abs(dp - dt)
    num = torch.sum(w * err, dim=1)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)
    return torch.mean(num / den)

def weighted_cosine_loss(dt: torch.Tensor, dp: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    w = gate_smoothstep(torch.abs(dt), a=GATE_A, b=GATE_B)
    num = torch.sum(w * dt * dp, dim=1)
    den = torch.clamp(torch.sqrt(torch.sum(w * dt * dt, dim=1)) * torch.sqrt(torch.sum(w * dp * dp, dim=1)), min=eps)
    cos = num / den
    return torch.mean(1.0 - cos)

def total_loss(dt: torch.Tensor, dp: torch.Tensor) -> torch.Tensor:
    return weighted_l1_like(dt, dp) + (LAMBDA_COS * weighted_cosine_loss(dt, dp))


In [26]:
# -----------------------------
# Model: bilinear + learned blending with baseline
# -----------------------------
class BilinearDeltaModelV2(nn.Module):
    def __init__(self, d_pert: int, d_out: int, rank_r: int, dropout: float):
        super().__init__()
        self.proj_p = nn.Sequential(
            nn.Linear(d_pert, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.proj_o = nn.Sequential(
            nn.Linear(d_out, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None

        self.scale = nn.Sequential(
            nn.Linear(d_pert, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = torch.nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = torch.nn.Parameter(torch.zeros(1, device=dev))


    def forward(self, z_pert: torch.Tensor, u_out: torch.Tensor, baseline: torch.Tensor) -> torch.Tensor:
        p = self.proj_p(z_pert)   # (B, R)
        o = self.proj_o(u_out)    # (G, R)
        y = p @ o.T               # (B, G)
        y = y + self.bias_gene[None, :] + self.bias_global

        s = self.scale(z_pert)    # (B, 1)
        y = (s * y) + ((1.0 - s) * baseline[None, :])
        return y


In [27]:
# -----------------------------
# Prepare tensors
# -----------------------------
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

Uo_t = torch.tensor(U_out, device=device)
Zt = torch.tensor(Z_train, device=device)
Yt = torch.tensor(Y, device=device)
baseline_t = torch.tensor(delta_baseline.astype(np.float32), device=device)

print("N:", N, "G:", G, "device:", device)


N: 80 G: 5127 device: cuda


In [28]:
# -----------------------------
# CV training (gene-level), early stop on official metric
# -----------------------------
def train_one_fold(tr_idx, va_idx):
    model = BilinearDeltaModelV2(EMB_DIM_PERT, EMB_DIM_OUT, RANK_R, DROPOUT).to(device)
    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    best_score = -1e18
    best_state = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t, baseline_t)
            loss = total_loss(Yt.index_select(0, b_t), pred)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo_t, baseline_t).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]
            s = score_delta(va_true, va_pred)
            sc = s["score"]

            if sc > best_score:
                best_score = sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_state

kf = KFold(n_splits=8, shuffle=True, random_state=SEED)
fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
    best_score, _ = train_one_fold(tr_idx, va_idx)
    fold_scores.append(float(best_score))
    print(f"fold {fold}: best_score={best_score:.6f}")

print(f"[cv] mean={float(np.mean(fold_scores)):.6f} std={float(np.std(fold_scores)):.6f}")


fold 1: best_score=0.087415
fold 2: best_score=0.063244
fold 3: best_score=0.065204
fold 4: best_score=0.069623
fold 5: best_score=0.072274
fold 6: best_score=0.106498
fold 7: best_score=0.113228
fold 8: best_score=0.056752
[cv] mean=0.079280 std=0.019572


In [29]:
# -----------------------------
# Fit final model on all 80, write submission
# -----------------------------
def fit_full_model():
    model = BilinearDeltaModelV2(EMB_DIM_PERT, EMB_DIM_OUT, RANK_R, DROPOUT).to(device)
    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    best_score = -1e18
    best_state = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t, baseline_t)
            loss = total_loss(Yt.index_select(0, b_t), pred)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt, Uo_t, baseline_t).detach().cpu().numpy().astype(np.float32)

            s = score_delta(Y, pred_np)
            sc = s["score"]
            print(f"epoch={epoch:4d} train_score={sc:.6f} wcos={s['wcos']:.6f} pred_wmae={s['pred_wmae']:.6f}")

            if sc > best_score:
                best_score = sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model

model_final = fit_full_model()

def predict_delta_gene(gene_symbol: str) -> np.ndarray:
    z = torch.tensor(emb(gene_symbol, EMB_DIM_PERT)[None, :].astype(np.float32), device=device)
    with torch.no_grad():
        y = model_final(z, Uo_t, baseline_t).detach().cpu().numpy().astype(np.float32)[0]
    return y

sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)
sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

idx = {g: i for i, g in enumerate(gene_columns)}
perm = [idx[g] for g in sub_gene_cols]

# default fill
sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

hit = 0
for pid, gene in val_map.items():
    vec = predict_delta_gene(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

print(f"[ok] filled {hit} leaderboard rows from mapping")

out_path = ROOT / "submission_bilinear_v2.csv"
sub.to_csv(out_path, index=False)
print("[ok] wrote:", out_path)


epoch=  25 train_score=0.102649 wcos=0.545536 pred_wmae=0.080925
epoch=  50 train_score=0.157888 wcos=0.565874 pred_wmae=0.077417
epoch=  75 train_score=0.188964 wcos=0.581836 pred_wmae=0.075641
epoch= 100 train_score=0.201433 wcos=0.596203 pred_wmae=0.075141
epoch= 125 train_score=0.210967 wcos=0.609414 pred_wmae=0.074778
epoch= 150 train_score=0.215958 wcos=0.619849 pred_wmae=0.074675
epoch= 175 train_score=0.223272 wcos=0.628788 pred_wmae=0.074389
epoch= 200 train_score=0.225014 wcos=0.636093 pred_wmae=0.074450
epoch= 225 train_score=0.232556 wcos=0.642738 pred_wmae=0.074112
epoch= 250 train_score=0.236514 wcos=0.649427 pred_wmae=0.074000
epoch= 275 train_score=0.240195 wcos=0.653946 pred_wmae=0.073869
epoch= 300 train_score=0.246123 wcos=0.659940 pred_wmae=0.073617
epoch= 325 train_score=0.248758 wcos=0.665110 pred_wmae=0.073585
epoch= 350 train_score=0.253086 wcos=0.669864 pred_wmae=0.073399
epoch= 375 train_score=0.257593 wcos=0.673878 pred_wmae=0.073230
epoch= 400 train_score=0.

KeyboardInterrupt: 

epoch=  25 train_score=0.098628 wcos=0.561587 pred_wmae=0.081470
epoch=  50 train_score=0.142988 wcos=0.616887 pred_wmae=0.079203
epoch=  75 train_score=0.182569 wcos=0.679437 pred_wmae=0.077505
epoch= 100 train_score=0.188548 wcos=0.734667 pred_wmae=0.077736
epoch= 125 train_score=0.197916 wcos=0.767679 pred_wmae=0.077553
epoch= 150 train_score=0.215600 wcos=0.787815 pred_wmae=0.076799
epoch= 175 train_score=0.243263 wcos=0.801258 pred_wmae=0.075532
epoch= 200 train_score=0.274973 wcos=0.811098 pred_wmae=0.074058
epoch= 225 train_score=0.311271 wcos=0.818643 pred_wmae=0.072328
epoch= 250 train_score=0.349535 wcos=0.825364 pred_wmae=0.070544
epoch= 275 train_score=0.392694 wcos=0.830588 pred_wmae=0.068472
epoch= 300 train_score=0.438821 wcos=0.833988 pred_wmae=0.066226
epoch= 325 train_score=0.482571 wcos=0.837453 pred_wmae=0.064087
epoch= 350 train_score=0.527659 wcos=0.840347 pred_wmae=0.061922
epoch= 375 train_score=0.568674 wcos=0.842771 pred_wmae=0.060007
epoch= 400 train_score=0.604465 wcos=0.844639 pred_wmae=0.058336
epoch= 425 train_score=0.632819 wcos=0.847145 pred_wmae=0.057115
epoch= 450 train_score=0.659838 wcos=0.849162 pred_wmae=0.055933
epoch= 475 train_score=0.681727 wcos=0.851390 pred_wmae=0.055034
epoch= 500 train_score=0.702127 wcos=0.853506 pred_wmae=0.054189
epoch= 525 train_score=0.720823 wcos=0.854851 pred_wmae=0.053418
epoch= 550 train_score=0.735669 wcos=0.856831 pred_wmae=0.052848
epoch= 575 train_score=0.750091 wcos=0.858768 pred_wmae=0.052284
epoch= 600 train_score=0.763336 wcos=0.859895 pred_wmae=0.051757
epoch= 625 train_score=0.775327 wcos=0.861221 pred_wmae=0.051295
epoch= 650 train_score=0.786127 wcos=0.862573 pred_wmae=0.050882
epoch= 675 train_score=0.797338 wcos=0.863707 pred_wmae=0.050463
epoch= 700 train_score=0.806733 wcos=0.864675 pred_wmae=0.050098
epoch= 725 train_score=0.815210 wcos=0.865884 pred_wmae=0.049795
epoch= 750 train_score=0.824120 wcos=0.866843 pred_wmae=0.049459
epoch= 775 train_score=0.831177 wcos=0.867719 pred_wmae=0.049204
epoch= 800 train_score=0.836577 wcos=0.868616 pred_wmae=0.049007
epoch= 825 train_score=0.845640 wcos=0.869823 pred_wmae=0.048698
epoch= 850 train_score=0.854120 wcos=0.870421 pred_wmae=0.048397
epoch= 875 train_score=0.859364 wcos=0.871047 pred_wmae=0.048199
epoch= 900 train_score=0.866102 wcos=0.872326 pred_wmae=0.047986
epoch= 925 train_score=0.871702 wcos=0.873291 pred_wmae=0.047828
epoch= 950 train_score=0.876338 wcos=0.873588 pred_wmae=0.047655
epoch= 975 train_score=0.880955 wcos=0.874203 pred_wmae=0.047495
epoch=1000 train_score=0.886360 wcos=0.874913 pred_wmae=0.047311
epoch=1025 train_score=0.891649 wcos=0.875464 pred_wmae=0.047137
epoch=1050 train_score=0.895180 wcos=0.876167 pred_wmae=0.047024
epoch=1075 train_score=0.897788 wcos=0.876503 pred_wmae=0.046941
epoch=1100 train_score=0.904590 wcos=0.876796 pred_wmae=0.046689
epoch=1125 train_score=0.907048 wcos=0.877637 pred_wmae=0.046632
epoch=1150 train_score=0.910048 wcos=0.878215 pred_wmae=0.046532
epoch=1175 train_score=0.911864 wcos=0.878767 pred_wmae=0.046495
epoch=1200 train_score=0.916365 wcos=0.879056 pred_wmae=0.046329
[ok] filled 60 leaderboard rows from mapping
[ok] wrote: submission_bilinear_v2.csv